In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "backend"))

from app.core.config import get_settings
from app.ranking.data import DataLoader
from app.ranking.profile.build_profile import recipe_string_to_user_vector, session_to_user_vector, _build_channel_to_aromas
from app.ranking.scoring.score import build_sku_vectors, score_skus

In [ ]:
settings = get_settings()
loader = DataLoader(settings.data_perfume_dir, settings.data_organ_dir if settings.data_organ_dir.exists() else None)

perfume_notes = loader.load_perfume_notes()
perfume_vectors, note_to_idx, idx_to_note = build_sku_vectors(perfume_notes)
print(f"SKU: {len(perfume_vectors)}, notes: {len(note_to_idx)}")

In [ ]:
recipe = "0:49,1:80,2:50,3:40,4:63,5:50"

aroma_map = loader.load_organ_aroma_notes_map()
channel_to_aromas = _build_channel_to_aromas(loader)
print("channel -> aroma:", channel_to_aromas)

user_vec = recipe_string_to_user_vector(recipe, aroma_map, channel_to_aromas=channel_to_aromas)
print(f"\nuser vector ({len(user_vec)} notes):", user_vec)

In [ ]:
perfumes = loader.load_perfumes()
popularity = perfumes.set_index("perfume_id")["allVotes"].to_dict() if "allVotes" in perfumes.columns else None

recs = score_skus(user_vec, perfume_vectors, note_to_idx, idx_to_note, top_n=10,
                  popularity=popularity, retrieval_k=100, blend_alpha=0.5)
print("Top-10:")
for pid, sc in recs:
    print(f"  {pid}  {sc:.4f}")

if loader.has_organ_data():
    sessions = loader.load_organ_sessions()
    for _, row in sessions.head(3).iterrows():
        u = session_to_user_vector(int(row["session_id"]), loader)
        r = score_skus(u, perfume_vectors, note_to_idx, idx_to_note, top_n=5,
                       popularity=popularity, retrieval_k=100, blend_alpha=0.5)
        hit = row["target_perfume_id"] in [x[0] for x in r]
        print(f"  session {row['session_id']}: target={row['target_perfume_id']} top5={[x[0] for x in r]} {'HIT' if hit else ''}")